In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix
import pickle

In [2]:
with open('log_reg_model.pkl', 'rb') as f:
    model = pickle.load(f)

In [3]:
df = pd.read_csv('final_data/cleaned_2025.csv')
df = df.iloc[:, :-4]
df = df[[i for i in df.columns if 'income' not in i]]
df = df[[i for i in df.columns if 'original' not in i]]
df = df[[i for i in df.columns if 'qa' not in i]]
df = df.drop(columns=['fd_label', 'label_status', 'label_reason'])

# adding a leading zero to tract_id to match geoid
df['tract_id'] = df['tract_id'].apply(lambda x: f'0{x}')

df.head()

,tract_id,State,County,meta_dataset_version,meta_population_census_year,meta_tract_vintage,clean__Urban,clean__population_total,clean__occupied_housing_units_total,clean__GroupQuartersFlag,...,clean__PovertyRate,clean__MedianFamilyIncome,clean__TractLOWI,clean__TractHUNV,clean__TractSNAP,clean__LATracts_half,clean__LATracts1,clean__LATractsVehicle_20,clean__low_access_population_1_20,FD_2
0,01001020100,Alabama,Autauga County,2025,2020,2020,1.0,1775.0,646.0,0.0,...,17.4,76979.0,1316.0,35.0,100.0,1.0,1.0,0.0,1080.0,1.0
1,01001020200,Alabama,Autauga County,2025,2020,2020,1.0,2055.0,723.0,0.0,...,9.5,62331.0,1148.0,28.0,143.0,1.0,0.0,0.0,167.0,0.0
2,01001020300,Alabama,Autauga County,2025,2020,2020,1.0,3216.0,1291.0,0.0,...,7.1,84191.0,2350.0,84.0,111.0,1.0,1.0,0.0,612.0,1.0
3,01001020400,Alabama,Autauga County,2025,2020,2020,1.0,4246.0,1707.0,0.0,...,15.3,81136.0,2580.0,114.0,78.0,1.0,0.0,0.0,14.0,0.0
4,01001020501,Alabama,Autauga County,2025,2020,2020,1.0,4322.0,1778.0,0.0,...,3.5,108233.0,4114.0,9.0,53.0,1.0,0.0,0.0,102.0,0.0


In [4]:
df.isnull().sum()

tract_id                                  0
State                                     0
County                                    0
meta_dataset_version                      0
meta_population_census_year               0
meta_tract_vintage                        0
clean__Urban                              0
clean__population_total                   0
clean__occupied_housing_units_total       0
clean__GroupQuartersFlag                271
clean__NUMGQTRS                           0
clean__PCTGQTRS                         271
clean__TractKids                          0
clean__TractSeniors                       0
clean__TractWhite                         0
clean__TractBlack                         0
clean__TractAsian                         0
clean__TractNHOPI                         0
clean__TractAIAN                          0
clean__TractOMultir                       0
clean__TractHispanic                      0
clean__LowIncomeTracts                    0
clean__PovertyRate              

In [5]:
df["clean__GroupQuartersFlag"] = df["clean__GroupQuartersFlag"].fillna(df["clean__GroupQuartersFlag"].median())
df["clean__PCTGQTRS"] = df["clean__PCTGQTRS"].fillna(df["clean__PCTGQTRS"].median())

df["clean__PovertyRate"] = df["clean__PovertyRate"].fillna(df["clean__PovertyRate"].median())
df["clean__MedianFamilyIncome"] = df["clean__MedianFamilyIncome"].fillna(df["clean__MedianFamilyIncome"].median())

df["clean__TractLOWI"] = df["clean__TractLOWI"].fillna(df["clean__TractLOWI"].median())
df["clean__TractHUNV"] = df["clean__TractHUNV"].fillna(df["clean__TractHUNV"].median())
df["clean__TractSNAP"] = df["clean__TractSNAP"].fillna(df["clean__TractSNAP"].median())

df["clean__LATracts_half"] = df["clean__LATracts_half"].fillna(df["clean__LATracts_half"].median())
df["clean__LATracts1"] = df["clean__LATracts1"].fillna(df["clean__LATracts1"].median())
df["clean__LATractsVehicle_20"] = df["clean__LATractsVehicle_20"].fillna(df["clean__LATractsVehicle_20"].median())

df["FD_2"] = df["FD_2"].fillna(0)

In [6]:
preds = model.predict(df)

df["pred_prob"] = preds
df["pred_class"] = (preds >= 0.45).astype(int)

df["risk"] = pd.cut(
    df["pred_prob"],
    bins=[0, 0.33, 0.67, 1],
    labels=["low-risk", "medium-risk", "high-risk"],
    include_lowest=True
)

print(df["risk"].value_counts())

risk
low-risk       72762
medium-risk    11186
high-risk        171
Name: count, dtype: int64


In [7]:
y_true_test = df["FD_2"]
y_pred_test = df["pred_class"]

print("Accuracy :", accuracy_score(y_true_test, y_pred_test))
print(confusion_matrix(y_true_test, y_pred_test)) # rows = actual, cols = predicted

Accuracy : 0.8320593445000535
[[68582  2182]
 [11945  1410]]


In [8]:
df.head()

,tract_id,State,County,meta_dataset_version,meta_population_census_year,meta_tract_vintage,clean__Urban,clean__population_total,clean__occupied_housing_units_total,clean__GroupQuartersFlag,...,clean__TractHUNV,clean__TractSNAP,clean__LATracts_half,clean__LATracts1,clean__LATractsVehicle_20,clean__low_access_population_1_20,FD_2,pred_prob,pred_class,risk
0,01001020100,Alabama,Autauga County,2025,2020,2020,1.0,1775.0,646.0,0.0,...,35.0,100.0,1.0,1.0,0.0,1080.0,1.0,0.195472,0,low-risk
1,01001020200,Alabama,Autauga County,2025,2020,2020,1.0,2055.0,723.0,0.0,...,28.0,143.0,1.0,0.0,0.0,167.0,0.0,0.213231,0,low-risk
2,01001020300,Alabama,Autauga County,2025,2020,2020,1.0,3216.0,1291.0,0.0,...,84.0,111.0,1.0,1.0,0.0,612.0,1.0,0.184558,0,low-risk
3,01001020400,Alabama,Autauga County,2025,2020,2020,1.0,4246.0,1707.0,0.0,...,114.0,78.0,1.0,0.0,0.0,14.0,0.0,0.188411,0,low-risk
4,01001020501,Alabama,Autauga County,2025,2020,2020,1.0,4322.0,1778.0,0.0,...,9.0,53.0,1.0,0.0,0.0,102.0,0.0,0.244368,0,low-risk


In [9]:
df.to_csv('cleaned_2025_w_predictions.csv', index=False)